# Create monthly temperature heatmaps

This notebook processes ERA5 2 m temperature data for the Mont Blanc region and visualises monthly temperature patterns over time. It calculates area-mean monthly temperatures and creates a combined figure showing monthly temperature variation alongside the annual mean temperature trend.

In [ ]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load ERA5 2 m temperature data
ds = xr.open_dataset("2m_temp.nc")

# Select approximate Mont Blanc area
subset = ds.sel(
    longitude=slice(6.7, 7.1),
    latitude=slice(46.0, 45.7)
)

# Calculate spatial mean across the selected area
mean_temp = subset["t2m"].mean(dim=["latitude", "longitude"])

# Convert to dataframe and Celsius
df = mean_temp.to_dataframe().reset_index()
df["t2m"] = df["t2m"] - 273.15

# Extract year and month
df["year"] = df["valid_time"].dt.year
df["month"] = df["valid_time"].dt.month

In [ ]:
# Calculate monthly means
monthly = (
    df.groupby(["year", "month"], as_index=False)["t2m"]
    .mean()
)

# Prepare heatmap data: month on y-axis, year on x-axis
heatmap_data = monthly.pivot(index="month", columns="year", values="t2m")
heatmap_data = heatmap_data.reindex(index=range(1, 13))

# Calculate annual mean temperature
annual = (
    monthly.groupby("year", as_index=False)["t2m"]
    .mean()
)

years = heatmap_data.columns.to_list()
x_positions = np.arange(len(years))

# Match annual data order to heatmap years
annual = annual.set_index("year").reindex(years).reset_index()

In [ ]:
# Create combined heatmap and annual trend figure
fig = plt.figure(figsize=(16, 8))

gs = fig.add_gridspec(
    nrows=2,
    ncols=2,
    width_ratios=[40, 1.8],
    height_ratios=[3, 1.2],
    hspace=0.12,
    wspace=0.10
)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
cax = fig.add_subplot(gs[0, 1])

# Heatmap
im = ax1.imshow(
    heatmap_data,
    aspect="auto",
    cmap="RdBu_r",
    origin="lower"
)

ax1.set_yticks(np.arange(12))
ax1.set_yticklabels([
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
])

ax1.set_ylabel("Month")
ax1.set_title(
    "Mont Blanc Area Monthly 2 m Air Temperature and Annual Mean Trend",
    fontsize=18,
    pad=20
)
ax1.tick_params(axis="x", labelbottom=False)

cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Temperature (°C)")

# Annual mean line graph
ax2.plot(x_positions, annual["t2m"], linewidth=1.8)

# Linear trend line
z = np.polyfit(x_positions, annual["t2m"], 1)
p = np.poly1d(z)
ax2.plot(x_positions, p(x_positions), linestyle="--", linewidth=1.5)

ax2.set_ylabel("Annual mean (°C)")
ax2.set_xlabel("Year")
ax2.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

step = max(1, len(years) // 12)
ax2.set_xticks(x_positions[::step])
ax2.set_xticklabels(years[::step], rotation=45, ha="right")

plt.show()

In [1]:
#save as 

fig.savefig("mont_blanc_monthly_temperature_heatmap.png", dpi=300, bbox_inches="tight")